# Role participation and role switching

Per-role participation distributions and the role-switching (co-membership) matrices for the
representative N4P2 / **ALL** trial at the post-trained checkpoint. A co-membership matrix is a
set operation over one population of neurons, so a single representative trial is used (pooling
or averaging would conflate realisations). Roles are lag-ordered: **L** (low-level), **H**
(high-level), **B** (binding); a PNG's anchor layer *l* is its H neuron's layer.

**Dependencies:**

Build the canonical HFB annotation tables for the representative trial first:
```bash
./scripts/analysis/build_annotation_table.py ./experiments/n4p2/train_n4p2_lrate_0_02_181023 ALL -v
```

**Plots:**

- **(A)** Per-role participation histograms (3x3 grid: roles L/H/B x anchor layers 2-4); the
  mean of each panel is marked.
- **(B)** Role-switching matrices - row-conditional co-membership proportions per anchor layer;
  rows are "fills role", columns "...also fills role"; structurally-impossible cells are greyed.

In [ ]:
from collections.abc import Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colors as mcolors
from mpl_toolkits.axes_grid1 import ImageGrid

from hsnn import viz
from hsnn.pipeline import reuse
from hsnn.pipeline.reuse.comembership import global_role_sets, role_comembership
from hsnn.utils import io

viz.setup_journal_env()

In [ ]:
# === CONFIGURATION ===
EXPERIMENT = "n4p2/train_n4p2_lrate_0_02_181023"  # reference combination
MODEL_TYPE = "ALL"                                # FF + LAT + FB architecture
CHECKPOINT = -1                                   # post-trained (last) state
ANCHOR_LAYERS = (2, 3, 4)                           # displayed (layer 1 not drawn)
ROLES = ("L", "H", "B")
ROLE_COL = {"L": "l_id", "H": "h_id", "B": "b_id"}
ROLE_LABEL = {"L": "Low-level (L)", "H": "High-level (H)", "B": "Binding (B)"}
ROLE_COLOUR = {"L": "C0", "H": "C1", "B": "C2"}

OUTPUT_DIR = io.BASE_DIR / "out/figures/supplementary/fig_S8"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def participation_counts(annotations: pd.DataFrame, anchor_layer: int, role: str) -> pd.Series:
    """Per-neuron participation counts for one role within an anchor layer."""
    subset = annotations.loc[annotations["layer"] == anchor_layer, ROLE_COL[role]]
    return subset.value_counts().sort_values(ascending=False)

## Load the representative-trial annotations

A neuron is identified by its composite `(layer, neuron)` key: the same physical neuron
regardless of which role column (`l_id`/`h_id`/`b_id`) it appears in.

In [ ]:
store = reuse.open_store(EXPERIMENT, MODEL_TYPE, checkpoint=CHECKPOINT)
rep_ann = reuse.load_tables(store)["hfb_annotations"]
REP_TRIAL = rep_ann["trial_id"].iloc[0]

print(f"Representative trial: {REP_TRIAL} | {len(rep_ann)} significant PNGs")
print("PNGs per anchor layer:")
print(rep_ann["layer"].value_counts().sort_index().to_string())

## Panel A - per-role participation distributions

A 3x3 grid of participation-count histograms: rows are the roles (L, H, B), columns the anchor
layers (2, 3, 4). For anchor-layer column *l*, the H and B histograms are for neurons in layer
*l* and the L histogram for neurons in layer *l*-1. The mean of each panel is marked.

In [ ]:
def plot_participation_grid(annotations: pd.DataFrame, layers: Sequence[int] = ANCHOR_LAYERS,
                            roles: Sequence[str] = ROLES, out_path=None):
    """3x3 grid of per-neuron participation histograms (rows=roles, cols=anchor layers)."""
    fig, axes = plt.subplots(len(roles), len(layers), figsize=(7.0, 4.0),
                             sharex="all", sharey="all")
    records = []
    for i, role in enumerate(roles):
        for j, layer in enumerate(layers):
            ax = axes[i, j]
            counts = participation_counts(annotations, layer, role)
            vals = counts.to_numpy()
            neuron_layer = layer - 1 if role == "L" else layer
            bins = np.arange(0.5, vals.max() + 1.5, 1)
            ax.hist(vals, bins=bins, edgecolor="black", linewidth=0.4, color=ROLE_COLOUR[role])
            ax.axvline(vals.mean(), color="black", ls="--", lw=1.5)
            ax.text(0.96, 0.92, f"$\\mu$={vals.mean():.1f}\nn={len(vals)}",
                    transform=ax.transAxes, ha="right", va="top", fontsize="x-small")
            if i == 0:
                ax.set_title(f"Layer {layer}", fontweight="bold")
            if j == 0:
                ax.set_ylabel(f"{ROLE_LABEL[role]}\n# neurons")
            if i == len(roles) - 1:
                ax.set_xlabel("PNGs per neuron")
            for neuron, c in counts.items():
                records.append({"layer": int(layer), "role": role,
                                "neuron_layer": int(neuron_layer), "neuron": int(neuron),
                                "participation": int(c)})
    axes[-1, -1].set_xlim(0, 20)
    fig.tight_layout()
    if out_path is not None:
        viz.save_figure(fig, out_path, overwrite=False)
    return fig, pd.DataFrame(records)


_, participation_data = plot_participation_grid(
    rep_ann, out_path=OUTPUT_DIR / "s8_participation_distributions.pdf")
participation_data.to_csv(OUTPUT_DIR / "s8_participation_distributions.csv", index=False)
print(f"Representative trial {REP_TRIAL}: {len(participation_data)} neuron-role rows")
plt.show()

## Panel B - role-switching matrices

Per-anchor-layer small multiples (layers 2-4), drawn as annotated heatmaps of the row-conditional
co-membership proportions on a shared `[0, 1]` colour scale. Rows are "fills role", columns
"...also fills role"; the diagonal is 1.0; the layer-4 low-level-column off-diagonals are
structurally impossible and greyed. A neuron's role repertoire is assessed over all significant
PNGs (including layer-1 circuits), so the off-diagonals are true switching rates.

In [ ]:
def plot_role_switching(matrices: Mapping[int, pd.DataFrame],
                        masks: Mapping[int, pd.DataFrame],
                        roles: Sequence[str] = ROLES,
                        cmap: str = "viridis", out_path=None) -> plt.Figure:
    """Small-multiple annotated heatmaps of role-switching proportions."""
    layers = list(matrices)
    base = plt.get_cmap(cmap).copy()
    base.set_bad("0.85")  # greyed structurally-impossible cells
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)
    n = len(roles)
    fig = plt.figure(figsize=(7.0, 2.4))
    grid = ImageGrid(fig, 111, nrows_ncols=(1, len(layers)), axes_pad=0.15,
                     share_all=True, cbar_mode="single", cbar_location="right",
                     cbar_size="5%", cbar_pad=0.1)
    im = None
    for ax, layer in zip(grid, layers):
        mat = matrices[layer].reindex(index=roles, columns=roles)
        mask = masks[layer].reindex(index=roles, columns=roles).to_numpy()
        data = np.ma.masked_array(mat.to_numpy(dtype=float), mask=mask)
        im = ax.imshow(data, cmap=base, norm=norm)
        for i in range(n):
            for j in range(n):
                if mask[i, j]:
                    ax.text(j, i, "—", ha="center", va="center", color="0.4",
                            fontsize="small")
                    continue
                val = mat.iloc[i, j]
                colour = "white" if val < 0.6 else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=colour,
                        fontsize="small")
        ax.set_xticks(range(n)); ax.set_yticks(range(n))
        ax.set_xticklabels(roles); ax.set_yticklabels(roles)
        ax.set_title(f"Layer {layer}", fontweight="bold")
        ax.tick_params(length=0)
    cb = grid.cbar_axes[0].colorbar(im)
    cb.set_label("Row-conditional proportion")
    if out_path is not None:
        viz.save_figure(fig, out_path, overwrite=False)
    return fig


global_sets = global_role_sets(rep_ann)
results = {L: role_comembership(rep_ann, L, global_sets=global_sets, top_layer=max(ANCHOR_LAYERS))
           for L in ANCHOR_LAYERS}
matrices = {L: r.proportions for L, r in results.items()}
masks = {L: r.mask for L, r in results.items()}

plot_role_switching(matrices, masks, out_path=OUTPUT_DIR / "role_switching_matrices.pdf")

# Tidy backing data: per anchor layer and ordered role pair.
sw_rows = []
for L in ANCHOR_LAYERS:
    res = results[L]
    for r1 in ROLES:
        for r2 in ROLES:
            sw_rows.append({
                "anchor_layer": L, "role_from": r1, "role_to": r2,
                "proportion": float(res.proportions.loc[r1, r2]),
                "comembership_count": int(res.counts.loc[r1, r2]),
                "row_pop_from": int(res.row_counts[r1]),
                "structurally_impossible": bool(res.mask.loc[r1, r2]),
            })
pd.DataFrame(sw_rows).to_csv(OUTPUT_DIR / "role_switching_matrices.csv", index=False)
plt.show()